# Generic Utils
> This module handles all data/model/trainer loading logic based on the pipeline config. It provides a clean interface for the main training script, abstracting away the details of how different components are instantiated.

In [ ]:
#| default_exp utils.__init__

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from fastcore import *
from fastcore.utils import *

In [1]:
#| export
from omegaconf import OmegaConf, DictConfig
import hydra
import torch
from torch import nn
from einops import rearrange
import math

In [ ]:
#| export
def init_data(cfg: DictConfig):
    """Instantiates the correct datamodule based on the pipeline config."""
    print(f"Initializing Datamodule: {cfg.pipeline.datamodule._target_}")
    return hydra.utils.instantiate(cfg.pipeline.datamodule)


In [ ]:
#| export
def init_model(cfg: DictConfig):
    """
    Instantiates the model(s).
    Returns a single model for Stage 1, or a dictionary of 3 models for Stage 2.
    """
    stage = cfg.pipeline.stage
    model_cfg = cfg.pipeline.model

    if stage == 1:
        # Stage 1: Simply instantiate and return the single VQ-VAE model
        return hydra.utils.instantiate(model_cfg)

    elif stage == 2:
        # Stage 2: Instantiate all three models independently
        models = {
            "power_net": hydra.utils.instantiate(model_cfg.power_net),
            "jepa": hydra.utils.instantiate(model_cfg.jepa)
        }
        
        # Instantiate VQ-VAE, then immediately load its pretrained weights and freeze it
        vqvae = hydra.utils.instantiate(model_cfg.vqvae)
        
        if hasattr(model_cfg.vqvae, "checkpoint_path") and model_cfg.vqvae.checkpoint_path:
            print(f"Loading pretrained VQ-VAE weights from {model_cfg.vqvae.checkpoint_path}")
            checkpoint = torch.load(model_cfg.vqvae.checkpoint_path, map_location="cpu")
            vqvae.load_state_dict(checkpoint["model_state_dict"]) 
            
        # Freeze VQ-VAE because it's only used for getting latent codebooks
        vqvae.eval()
        for param in vqvae.parameters():
            param.requires_grad = False
            
        models["vqvae"] = vqvae
        return models
    
    else:
        raise ValueError(f"Unknown pipeline stage: {stage}")



In [ ]:
#| export
def init_trainer(cfg: DictConfig, data_module, models, device, slurm_jobid):
    """Instantiates the trainer and injects the loaded models and data."""
    # We pass models and datamodule directly into the instantiation call 
    # so the trainer receives its dependencies right away.
    return hydra.utils.instantiate(
        cfg.pipeline.trainer, 
        data_module=data_module,
        model=models, 
        device=device,
        slurm_jobid=slurm_jobid
    )


In [ ]:
#| export
@torch.no_grad()
def channel(schedule, power, msg_indices, csi, device, codebook_size=256, snr_db=10.0, no_comm= False):
    """
    schedule:    (B*T, n, 1)
    power:       (B*T, n, 1)
    msg_indices: (B*T, 49)
    csi:         (B*T, n, 1) complex
    returns:     (B*T, 49) recovered indices (per neighbor averaged or selected)
    """
    if schedule is None or power is None:
        return msg_indices

    symbol_length = int(math.log2(codebook_size))  # 8 bits for k=256
    BT, msg_dim = msg_indices.shape
    n = csi.shape[1]

    # Expand message for each neighbor: (B*T, n, 49)
    msg_expanded = msg_indices.unsqueeze(1).expand(-1, n, -1)
    msg_flat = msg_expanded.reshape(BT * n, msg_dim)           # (B*T*n, 49)

    # Flatten neighbor dim for schedule/power/csi
    schedule_flat = rearrange(schedule, "b n d -> (b n) d")    # (B*T*n, 1)
    power_flat    = rearrange(power,    "b n d -> (b n) d")    # (B*T*n, 1)
    csi_flat      = rearrange(csi,      "b n d -> (b n) d")    # (B*T*n, 1) complex

    # 1. Binarize — vectorized
    bit_shifts = torch.arange(symbol_length, device=device)       # (8,)
    bits = ((msg_flat.unsqueeze(-1) >> bit_shifts) & 1).float()        # (B*T*n, 49, 8)
    bits = bits.reshape(BT * n, msg_dim * symbol_length)               # (B*T*n, 392)

    # 2. BPSK modulation: 0 → -1, 1 → +1
    modulated = 2 * bits - 1                                           # (B*T*n, 392)

    # 3. Power scaling
    modulated = modulated * torch.sqrt(power_flat)                     # (B*T*n, 392)

    # 4. Channel effect + AWGN
    snr_linear = 10 ** (snr_db / 10.0)
    noise_std = torch.sqrt(power_flat / snr_linear)                    # (B*T*n, 1)
    noise = (torch.randn_like(modulated) + 1j * torch.randn_like(modulated)) * noise_std / math.sqrt(2)
    received = modulated * csi_flat + noise    # (B*T*n, 392) complex
    # Note: using real part of csi for BPSK (real-valued modulation)

    # 5. Matched filter demodulation
    demodulated = ((received * torch.conj(csi_flat)).real > 0).float() # (B*T*n, 392)

    # 6. De-binarize — vectorized
    bit_weights = (2 ** bit_shifts).long()                             # (8,)
    recovered = (
        demodulated.reshape(BT * n, msg_dim, symbol_length).long()
        * bit_weights
    ).sum(-1)                                                           # (B*T*n, 49)

    # 7. Apply schedule: if not scheduled, receiver gets no message
    # Fall back to original indices (no information case)
    ##If schedule=1 (send): recovered = 1 * recovered + 0 * msg_flat → use the channel-degraded received message 
    # If schedule=0 (don't send): recovered = 0 * recovered + 1 * msg_flat → use original indices
    
    schedule_mask = (schedule_flat > 0.5)                   # (B*T*n, 1) bool

    # Where not scheduled, use a special null index (e.g. codebook_size as padding token)
    null_indices = torch.full_like(msg_flat, codebook_size)  # out-of-vocab index shape: (B*T*n, 49)
    if no_comm:
        recovered = null_indices
        recovered = recovered.reshape(BT, n, msg_dim)
        return recovered[:, 0, :]
    
    recovered = torch.where(schedule_mask, recovered, null_indices)

    # schedule_mask = (schedule_flat > 0.5).long()                       # (B*T*n, 1)
    # recovered = schedule_mask * recovered + (1 - schedule_mask) * msg_flat

    # 8. Reshape back: (B*T, n, 49)
    recovered = recovered.reshape(BT, n, msg_dim)

    # 9. Aggregate across neighbors
    # Simple approach: take the first neighbor (the intended one)
    # Or average if you want to combine information from multiple neighbors
    recovered = recovered[:, 0, :]                                     # (B*T, 49)

    return recovered

In [31]:
#| hide
# schedule, power, msg_indices, csi_flat, device=self.device
n_neighbors = 1
B = 32
T = 3
schedule = torch.rand(B*T, n_neighbors, 1)
power = torch.rand(B*T, n_neighbors, 1)
msg_indices = torch.randint(0, 256, (B*T, 49))
csi = torch.rand(B*T, n_neighbors, 1) + 1
device = "cpu"
received_msg = channel(schedule, power, msg_indices, csi, device=device)
print(received_msg.shape)

torch.Size([96, 49])


In [32]:
#| hide
received_msg

tensor([[256, 256, 256,  ..., 256, 256, 256],
        [107,  71, 134,  ..., 247,  42, 124],
        [  2,  98,  48,  ..., 242,  46,  33],
        ...,
        [125, 107,  72,  ..., 196, 164, 130],
        [251, 102,  25,  ...,  22,  14, 172],
        [214,  66, 125,  ..., 218, 184, 159]])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()